# DCF Valuation Walkthrough
### EquityIQ Research — Methodology Deep Dive

This notebook walks through a complete Discounted Cash Flow (DCF) valuation for an NSE-listed company from scratch — the same methodology implemented in the EquityIQ platform.

**Ticker used:** INFY (Infosys Limited)

---

## 1. Intrinsic vs Relative Valuation

There are two fundamental approaches to valuing a company:

| Approach | Method | Question it answers |
|---|---|---|
| **Intrinsic** | DCF | What is this company *worth* based on its cash flows? |
| **Relative** | Comps | What is the *market paying* for similar companies? |

A DCF is capital-structure neutral — it values the **business**, not the equity. We get to equity value at the end by subtracting net debt.

**Why FCFF over Net Income?**
- Net Income includes interest expense → affected by how the company is financed
- Net Income includes non-cash items (D&A) → not the same as cash
- FCFF = cash available to ALL capital providers (debt + equity), after reinvestment needs

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

TICKER = 'INFY'
t = yf.Ticker(f'{TICKER}.NS')
info = t.info

print(f"Company  : {info.get('longName')}")
print(f"Sector   : {info.get('sector')}")
print(f"Industry : {info.get('industry')}")
print(f"Price    : ₹{info.get('currentPrice', 0):,.2f}")
print(f"Mkt Cap  : ₹{info.get('marketCap', 0)/1e7:,.0f} Crores")

## 2. Loading Financial Statements

All values are in **₹ Crores** (1 Crore = 10,000,000 INR).

In [ ]:
SCALE = 1e7  # Convert raw INR to Crores

income = t.income_stmt.iloc[:, :5] / SCALE
balance = t.balance_sheet.iloc[:, :5] / SCALE
cashflow = t.cash_flow.iloc[:, :5] / SCALE

print('=== INCOME STATEMENT (₹ Cr) ===')
key_is = ['Total Revenue', 'EBIT', 'Net Income', 'Pretax Income']
display(income.loc[[r for r in key_is if r in income.index]].round(0))

print('\n=== BALANCE SHEET (₹ Cr) ===')
key_bs = ['Total Assets', 'Stockholders Equity', 'Total Debt', 'Cash And Cash Equivalents']
display(balance.loc[[r for r in key_bs if r in balance.index]].round(0))

## 3. Computing Historical FCFF

$$FCFF = EBIT \times (1 - t) + D\&A - CapEx - \Delta WC$$

Where:
- $EBIT \times (1-t)$ = **NOPAT** — Net Operating Profit After Tax (no interest effect)
- $D\&A$ — added back because it's non-cash
- $CapEx$ — subtracted because it's real cash out
- $\Delta WC$ — change in working capital (cash tied up in operations)

In [ ]:
def get_safe(df, labels, col=0):
    for label in labels:
        if label in df.index:
            val = df.iloc[df.index.get_loc(label), col]
            if pd.notna(val):
                return float(val)
    return 0.0

years = [col.year for col in income.columns]
records = []

for i in range(len(years)):
    revenue  = get_safe(income,   ['Total Revenue', 'TotalRevenue'], i)
    ebit     = get_safe(income,   ['EBIT', 'Operating Income'], i)
    pretax   = get_safe(income,   ['Pretax Income', 'Income Before Tax'], i)
    net_inc  = get_safe(income,   ['Net Income', 'NetIncome'], i)
    da       = abs(get_safe(cashflow, ['Depreciation And Amortization', 'Depreciation'], i))
    capex    = abs(get_safe(cashflow, ['Capital Expenditure', 'Purchase Of Property Plant And Equipment'], i))
    delta_wc = get_safe(cashflow, ['Change In Working Capital', 'Changes In Working Capital'], i)

    tax_rate = max(0.10, min(1 - (net_inc / pretax if pretax != 0 else 0.75), 0.40))
    nopat    = ebit * (1 - tax_rate)
    fcff     = nopat + da - capex - delta_wc

    records.append({
        'Year': years[i], 'Revenue': round(revenue, 0),
        'EBIT': round(ebit, 0), 'Tax Rate': f'{tax_rate*100:.1f}%',
        'NOPAT': round(nopat, 0), 'D&A': round(da, 0),
        'CapEx': round(capex, 0), 'Δ WC': round(delta_wc, 0),
        'FCFF': round(fcff, 0)
    })

hist_df = pd.DataFrame(records).set_index('Year')
print('=== HISTORICAL FCFF (₹ Cr) ===')
display(hist_df)

## 4. WACC Calculation

$$WACC = \frac{E}{V} \times R_e + \frac{D}{V} \times R_d \times (1-t)$$

**Cost of Equity via CAPM:**
$$R_e = R_f + \beta \times ERP$$

Where:
- $R_f$ = **7.0%** — India 10-year government bond yield
- $\beta$ = from yfinance (company's sensitivity to market)
- $ERP$ = **7.5%** — India Equity Risk Premium (Damodaran, 2024)

In [ ]:
Rf  = 0.07    # India 10Y Gsec yield
ERP = 0.075   # Damodaran India ERP

beta = info.get('beta') or 1.0
beta = max(0.3, min(beta, 2.5))

Re = Rf + beta * ERP

# Cost of debt
total_debt = get_safe(balance, ['Total Debt', 'Long Term Debt']) 
int_exp    = abs(get_safe(income, ['Interest Expense', 'InterestExpense']))
Rd = (int_exp / total_debt) if total_debt > 0 else 0.09
Rd = max(0.06, min(Rd, 0.18))

# Weights
mkt_cap = (info.get('marketCap') or 0) / 1e7
E, D    = mkt_cap, total_debt
V       = E + D
tax_r   = 0.25

WACC = (E/V) * Re + (D/V) * Rd * (1 - tax_r)

print(f'Beta              : {beta:.2f}')
print(f'Risk-Free Rate    : {Rf*100:.1f}%')
print(f'Equity Risk Prem  : {ERP*100:.1f}%')
print(f'Cost of Equity    : {Re*100:.2f}%  (Rf + β × ERP)')
print(f'Cost of Debt      : {Rd*100:.2f}%')
print(f'Equity Weight     : {E/V*100:.1f}%')
print(f'Debt Weight       : {D/V*100:.1f}%')
print(f'Tax Rate          : {tax_r*100:.0f}%')
print(f'─────────────────────────────')
print(f'WACC              : {WACC*100:.2f}%')

## 5. Projecting FCFF — Three Scenarios

We project 5 years of FCFF under three revenue growth assumptions:
- **Base**: Historical CAGR
- **Bull**: Base + 3%
- **Bear**: Base − 3%

All other assumptions (EBIT margin, tax rate, CapEx %, D&A %) are the historical averages.

In [ ]:
revenues = hist_df['Revenue'].values
cagr = (revenues[0] / revenues[-1]) ** (1/(len(revenues)-1)) - 1
cagr = max(0.03, min(cagr, 0.35))

avg_margin  = (hist_df['EBIT'] / hist_df['Revenue']).mean()
avg_capex   = min((hist_df['CapEx'] / hist_df['Revenue']).mean(), 0.08)
avg_da      = (hist_df['D&A'] / hist_df['Revenue']).mean()
avg_tax     = 0.25
base_rev    = float(revenues[0])

print(f'Base Revenue      : ₹{base_rev:,.0f} Cr')
print(f'Revenue CAGR      : {cagr*100:.1f}%')
print(f'EBIT Margin       : {avg_margin*100:.1f}%')
print(f'CapEx % of Rev    : {avg_capex*100:.1f}%')
print(f'D&A % of Rev      : {avg_da*100:.1f}%')

def project(growth, label):
    records = []
    prev_rev = base_rev
    for yr in range(1, 6):
        rev   = prev_rev * (1 + growth)
        ebit  = rev * avg_margin
        nopat = ebit * (1 - avg_tax)
        da    = rev * avg_da
        capex = rev * avg_capex
        dwc   = rev * 0.01
        fcff  = nopat + da - capex - dwc
        records.append({'Year': f'Y+{yr}', 'Revenue': round(rev,0),
                        'EBIT': round(ebit,0), 'FCFF': round(fcff,0)})
        prev_rev = rev
    df = pd.DataFrame(records).set_index('Year')
    print(f'\n=== {label} Case (Growth: {growth*100:.1f}%) ===')
    display(df)
    return df

base_proj = project(cagr,        'BASE')
bull_proj = project(cagr + 0.03, 'BULL')
bear_proj = project(cagr - 0.03, 'BEAR')

## 6. Terminal Value

We only explicitly project 5 years. The **Terminal Value** captures all cash flows from Year 6 to infinity:

$$TV = \frac{FCFF_5 \times (1+g)}{WACC - g}$$

$$PV(TV) = \frac{TV}{(1+WACC)^5}$$

**Terminal growth rate (g) = 4%** — slightly above India's long-run inflation, below GDP growth.

In [ ]:
g = 0.04  # Terminal growth rate

def dcf_value(proj_df, scenario_name):
    fcffs   = proj_df['FCFF'].values
    pv_fcff = sum(f / (1+WACC)**t for t, f in enumerate(fcffs, 1))
    tv      = fcffs[-1] * (1+g) / (WACC - g)
    pv_tv   = tv / (1+WACC)**5
    ev      = pv_fcff + pv_tv

    net_debt   = total_debt - get_safe(balance, ['Cash And Cash Equivalents', 'Cash Financial'])
    eq_val     = ev - net_debt
    shares     = (info.get('sharesOutstanding') or 0) / 1e7
    intr_price = eq_val / shares if shares > 0 else 0
    curr_price = info.get('currentPrice') or 0
    upside     = (intr_price - curr_price) / curr_price * 100 if curr_price else 0

    print(f'\n=== {scenario_name} Case DCF ===')
    print(f'PV of FCFFs       : ₹{pv_fcff:,.0f} Cr')
    print(f'Terminal Value    : ₹{tv:,.0f} Cr')
    print(f'PV of TV          : ₹{pv_tv:,.0f} Cr  ({pv_tv/ev*100:.0f}% of EV)')
    print(f'Enterprise Value  : ₹{ev:,.0f} Cr')
    print(f'Less: Net Debt    : ₹{net_debt:,.0f} Cr')
    print(f'Equity Value      : ₹{eq_val:,.0f} Cr')
    print(f'Shares (Cr)       : {shares:.1f}')
    print(f'Intrinsic Price   : ₹{intr_price:,.0f}')
    print(f'Current Price     : ₹{curr_price:,.0f}')
    print(f'Upside / Downside : {upside:+.1f}%')
    return intr_price

base_px = dcf_value(base_proj, 'BASE')
bull_px = dcf_value(bull_proj, 'BULL')
bear_px = dcf_value(bear_proj, 'BEAR')

## 7. Sensitivity Analysis

The most important output of any DCF — a grid of implied prices across different WACC and terminal growth rate combinations.

**Why?** Terminal value typically represents 60–80% of enterprise value. Small changes in WACC or g produce large swings in the output. Showing a range is more honest than a single number.

In [ ]:
fcffs_base = base_proj['FCFF'].values
net_debt   = total_debt - get_safe(balance, ['Cash And Cash Equivalents', 'Cash Financial'])
shares     = (info.get('sharesOutstanding') or 0) / 1e7
curr_price = info.get('currentPrice') or 0

wacc_range   = [WACC + d for d in [-0.02, -0.01, 0, 0.01, 0.02]]
growth_range = [g + d for d in [-0.01, -0.005, 0, 0.005, 0.01]]

grid = []
for w in wacc_range:
    row = []
    for gr in growth_range:
        if w <= gr:
            row.append(None)
            continue
        pv  = sum(f/(1+w)**t for t,f in enumerate(fcffs_base,1))
        tv  = fcffs_base[-1]*(1+gr)/(w-gr)
        ev  = pv + tv/(1+w)**5
        px  = (ev - net_debt) / shares if shares > 0 else 0
        row.append(round(px, 0))
    grid.append(row)

sens_df = pd.DataFrame(
    grid,
    index   = [f'{w*100:.1f}%' for w in wacc_range],
    columns = [f'{gr*100:.1f}%' for gr in growth_range]
)
sens_df.index.name   = 'WACC \ Growth'

fig = go.Figure(go.Heatmap(
    z    = [[v if v else 0 for v in row] for row in grid],
    x    = [f'{gr*100:.1f}%' for gr in growth_range],
    y    = [f'{w*100:.1f}%' for w in wacc_range],
    text = [[f'₹{v:,.0f}' if v else '—' for v in row] for row in grid],
    texttemplate = '%{text}',
    colorscale   = [[0,'#c62828'],[0.5,'#ffd700'],[1,'#2e7d32']],
    zmid         = curr_price,
    showscale    = True,
))
fig.update_layout(
    title  = f'{TICKER} — DCF Sensitivity: WACC × Terminal Growth (Implied Share Price)',
    xaxis_title = 'Terminal Growth Rate →',
    yaxis_title = 'WACC ↓',
    height = 400,
)
fig.show()
print(f'\nGreen = above current price ₹{curr_price:,.0f} (upside)')
print(f'Red   = below current price (downside)')

## 8. Recommendation

Based on our DCF across three scenarios:

In [ ]:
curr_price = info.get('currentPrice') or 0
avg_target = (base_px + bull_px + bear_px) / 3
upside     = (avg_target - curr_price) / curr_price * 100 if curr_price else 0

rating = 'BUY' if upside > 15 else ('SELL' if upside < -15 else 'HOLD')

print(f'Company       : {info.get("longName")}')
print(f'Current Price : ₹{curr_price:,.0f}')
print(f'Bear Target   : ₹{bear_px:,.0f}')
print(f'Base Target   : ₹{base_px:,.0f}')
print(f'Bull Target   : ₹{bull_px:,.0f}')
print(f'Avg Target    : ₹{avg_target:,.0f}')
print(f'Implied Return: {upside:+.1f}%')
print(f'─────────────────────────────')
print(f'RATING        : {rating}')

---

## Key Takeaways

1. **FCFF strips out financing** — it values the business independent of capital structure
2. **WACC is the hurdle rate** — the return the company must earn to create value
3. **Terminal value dominates** — typically 60–80% of enterprise value, making the growth rate assumption the most sensitive input
4. **Sensitivity tables are essential** — a single-point DCF output gives false precision; the range of outcomes matters more than the central estimate
5. **DCF + Comps = complete picture** — DCF tells you what it's worth, comps tell you what the market is paying

---

**Try the full EquityIQ platform:** Any NSE ticker, all methodologies, 6-page PDF report  
**GitHub:** https://github.com/divyamchoudhary8/equity-research-tool